In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\GUESTT\Desktop\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.0.4
c:\Users\GUESTT\Desktop\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [4]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [5]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"C:\Users\GUESTT\Desktop\Regression_ML_EndtoEnd\data\processed\feature_engineered_train.csv")
eval_df  = pd.read_csv(r"C:\Users\GUESTT\Desktop\Regression_ML_EndtoEnd\data\processed\feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [6]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [14]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("file:///C:/Users/GUESTT/Desktop/Regression_ML_EndtoEnd/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/05/07 14:30:24 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-05-07 14:30:24,538] A new study created in memory with name: no-name-ebfe2feb-c879-4a23-9527-cd5c484fe103


[I 2026-05-07 14:31:34,670] Trial 0 finished with value: 71898.90341605518 and parameters: {'n_estimators': 902, 'max_depth': 7, 'learning_rate': 0.047061872397888706, 'subsample': 0.5430930028129328, 'colsample_bytree': 0.8570089542042798, 'min_child_weight': 5, 'gamma': 1.3725885485359601, 'reg_alpha': 1.9876110325909393e-06, 'reg_lambda': 2.7182468764307703}. Best is trial 0 with value: 71898.90341605518.
[I 2026-05-07 14:33:25,177] Trial 1 finished with value: 72178.8596069032 and parameters: {'n_estimators': 620, 'max_depth': 10, 'learning_rate': 0.026577028872442232, 'subsample': 0.619555660828095, 'colsample_bytree': 0.9879639592777456, 'min_child_weight': 1, 'gamma': 1.9279645628634834, 'reg_alpha': 3.328297281958477e-07, 'reg_lambda': 8.320435019001026}. Best is trial 0 with value: 71898.90341605518.
[I 2026-05-07 14:34:06,888] Trial 2 finished with value: 87891.62509809366 and parameters: {'n_estimators': 796, 'max_depth': 3, 'learning_rate': 0.017493139711390666, 'subsample'

Best params: {'n_estimators': 988, 'max_depth': 10, 'learning_rate': 0.04218265585629907, 'subsample': 0.5099995166206827, 'colsample_bytree': 0.9644369613399694, 'min_child_weight': 8, 'gamma': 1.4881902110499077, 'reg_alpha': 6.6062865574994276e-06, 'reg_lambda': 9.457462599555228}


In [15]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 30276.139054186264
RMSE: 71323.92013710976
R²: 0.9606875200518095


c:\Users\GUESTT\Desktop\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [15:20:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/05/07 15:20:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/07 15:20:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
